# Sentiment–Rating Incongruence in Sri Lanka Tourism Attraction Reviews

| RQ | Question |
|----|----------|
| **RQ1** | How prevalent is incongruence, and what is the full typology of directional patterns? |
| **RQ2** | Does incongruence vary significantly across location types? |
| **RQ3** | Which reviewer-level and review-level characteristics are associated with incongruence? |



In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.pipeline import Pipeline
import statsmodels.api as sm
import warnings
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')
# ── Colour palette ──────────────────────────────────────────────────────
P = dict(main='#4A6FA5', acc='#E05C5C', neu='#F0A500',
         grn='#2E9E6B', light='#B8CCE4', purple='#7C3AED')
# ── Global plot defaults ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F7',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : '#E5E5E0',
    'grid.linewidth'   : 0.6,
    'font.family'      : 'sans-serif',
    'axes.titlesize'   : 12,
    'axes.titleweight' : '600',
    'axes.labelsize'   : 10,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
})

In [2]:
DATA_PATH = 'Processed_Reviews_with_Sentiment.csv'
df = pd.read_csv(DATA_PATH)
df['Sentiment'] = df['Sentiment'].str.strip().str.upper()

assert set(df['Rating_Class'].unique()).issubset({'Positive','Neutral','Negative'}), \
    'Unexpected Rating_Class values'
assert set(df['Sentiment'].unique()).issubset({'POSITIVE','NEUTRAL','NEGATIVE'}), \
    'Unexpected Sentiment values'

print(f'✓ Dataset loaded  : {df.shape[0]:,} reviews x {df.shape[1]} columns')
print(f'  Year range      : {df["Travel_Date_Year"].min()} - {df["Travel_Date_Year"].max()}')
print(f'  Location types  : {df["Location_Type"].nunique()}')
print(f'  User countries  : {df["User_Country"].nunique()}')
print(f'\n  Rating_Class distribution:')
print(df['Rating_Class'].value_counts().to_string())
print(f'\n  Sentiment distribution:')
print(df['Sentiment'].value_counts().to_string())
print(f'\n  Missing values:')
mv = df.isnull().sum()
print(mv[mv > 0].to_string() if mv.sum() > 0 else '  None')

✓ Dataset loaded  : 16,156 reviews x 21 columns
  Year range      : 2010 - 2023
  Location types  : 11
  User countries  : 140

  Rating_Class distribution:
Rating_Class
Positive    12845
Neutral      2166
Negative     1145

  Sentiment distribution:
Sentiment
POSITIVE    13041
NEGATIVE     1602
NEUTRAL      1513

  Missing values:
  None


## 2. Pre-analysis Visualisations
Four panels: star rating distribution, sentiment label distribution, review count by location type, and reviewer tier distribution.  
Then a standalone Rating Class × Sentiment heatmap showing the off-diagonal (incongruent) cells directly before statistical testing begins.

In [3]:
# ── Figure P1: Four-panel dataset overview ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Dataset Overview — Before Analysis', fontsize=14,
             fontweight='700', y=1.01)

# Panel A — Star rating distribution
ax = axes[0, 0]
rating_counts = df['Rating'].value_counts().sort_index()
bar_colors_r  = [P['acc'] if r <= 2 else (P['neu'] if r == 3 else P['grn'])
                 for r in rating_counts.index]
bars = ax.bar(rating_counts.index.astype(str), rating_counts.values,
              color=bar_colors_r, edgecolor='white', width=0.65)
for bar, val in zip(bars, rating_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Star Rating')
ax.set_ylabel('Number of Reviews')
ax.set_title('A. Star Rating Distribution\n(1–2: Negative | 3: Neutral | 4–5: Positive)', pad=8)
ax.set_ylim(0, rating_counts.max() * 1.22)
ax.legend(handles=[
    Patch(facecolor=P['acc'], label='Negative (1–2★)'),
    Patch(facecolor=P['neu'], label='Neutral (3★)'),
    Patch(facecolor=P['grn'], label='Positive (4–5★)')],
    frameon=False, fontsize=8)

# Panel B — Sentiment label distribution
ax = axes[0, 1]
sent_order  = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
sent_colors = [P['grn'], P['neu'], P['acc']]
sent_counts = df['Sentiment'].value_counts().reindex(sent_order)
bars = ax.bar(sent_order, sent_counts.values, color=sent_colors,
              edgecolor='white', width=0.55)
for bar, val in zip(bars, sent_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Sentiment Label (Cardiff RoBERTa)')
ax.set_ylabel('Number of Reviews')
ax.set_title('B. Sentiment Label Distribution\n(cardiffnlp/twitter-roberta-base-sentiment)', pad=8)
ax.set_ylim(0, sent_counts.max() * 1.22)

# Panel C — Review count by location type
ax = axes[1, 0]
lt_counts = df['Location_Type'].value_counts().sort_values()
ax.barh(lt_counts.index, lt_counts.values, color=P['main'],
        edgecolor='white', height=0.65)
for i, val in enumerate(lt_counts.values):
    ax.text(val + 30, i, f'{val:,}', va='center', fontsize=8.5)
ax.set_xlabel('Number of Reviews')
ax.set_title('C. Review Count by Location Type', pad=8)
ax.set_xlim(0, lt_counts.max() * 1.18)

# Panel D — Reviewer tier distribution
ax = axes[1, 1]
def tier_label(x):
    if pd.isna(x) or x < 0:
        return np.nan
    if x <= 5:
        return 'Novice (0-5)'
    elif x <= 20:
        return 'Casual (6-20)'
    elif x <= 100:
        return 'Active (21-100)'
    return 'Expert (101+)'

tier_labels = ['Novice (0-5)', 'Casual (6-20)', 'Active (21-100)', 'Expert (101+)']
tier_colors = [P['grn'], P['light'], P['main'], P['acc']]
tier_counts_raw = df['User_Contributions'].apply(tier_label).value_counts().reindex(tier_labels)
bars = ax.bar(tier_labels, tier_counts_raw.values,
              color=tier_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, tier_counts_raw.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=8.5)
ax.set_xlabel('Reviewer Tier')
ax.set_ylabel('Number of Reviews')
ax.set_title('D. Reviewer Tier Distribution\n(cut-points: 5, 20, 100 contributions)', pad=8)
ax.set_ylim(0, tier_counts_raw.max() * 1.22)
ax.tick_params(axis='x', labelsize=8)

plt.tight_layout(pad=2.5)
plt.savefig('fig_p1_dataset_overview.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure P1 saved: fig_p1_dataset_overview.png')

# ── Figure P2: Rating Class × Sentiment Heatmap ─────────────────────────
rating_order    = ['Positive', 'Neutral', 'Negative']
sentiment_order = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
ct_matrix  = pd.crosstab(df['Rating_Class'], df['Sentiment'])
ct_matrix  = ct_matrix.reindex(index=rating_order, columns=sentiment_order, fill_value=0)
pct_matrix = (ct_matrix / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 6))
for i in range(3):
    for j in range(3):
        count    = ct_matrix.values[i, j]
        pct      = pct_matrix.values[i, j]
        bg_color = '#D4EDDA' if i == j else '#FDECEA'
        rect     = plt.Rectangle([j-0.5, i-0.5], 1, 1,
                                  facecolor=bg_color, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        ax.text(j, i, f'{count:,}\n({pct}%)', ha='center', va='center', fontsize=11,
                fontweight='700' if i == j else '400',
                color='#1B5E20' if i == j else '#B71C1C')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['POSITIVE\nsentiment', 'NEUTRAL\nsentiment', 'NEGATIVE\nsentiment'], fontsize=10)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Positive\n(4–5★)', 'Neutral\n(3★)', 'Negative\n(1–2★)'], fontsize=10)
ax.set_xlabel('Text Sentiment (Cardiff RoBERTa)', fontsize=11, labelpad=10)
ax.set_ylabel('Star Rating Class', fontsize=11, labelpad=10)
ax.set_title('Rating Class × Sentiment Heatmap\nGreen diagonal = Congruent | Red off-diagonal = Incongruent',
             pad=12, fontweight='700')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(2.5, -0.5)
total_inc = sum(ct_matrix.values[i,j] for i in range(3) for j in range(3) if i != j)
total_pct_h = total_inc / len(df) * 100
ax.text(0.5, -0.18,
        f'Total incongruent: {total_inc:,} reviews ({total_pct_h:.1f}% of all reviews)',
        transform=ax.transAxes, ha='center', fontsize=10, color='#B71C1C', fontweight='600')
ax.legend(handles=[
    Patch(facecolor='#D4EDDA', edgecolor='#2E7D32', label='Congruent (diagonal)'),
    Patch(facecolor='#FDECEA', edgecolor='#C62828', label='Incongruent (off-diagonal)')],
    loc='upper right', bbox_to_anchor=(1.0, -0.14), frameon=False, ncol=2, fontsize=9)
plt.tight_layout(pad=2)
plt.savefig('fig_p2_heatmap.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure P2 saved: fig_p2_heatmap.png')
print(f'  Congruent   : {len(df)-total_inc:,}  ({(len(df)-total_inc)/len(df)*100:.1f}%)')
print(f'  Incongruent : {total_inc:,}  ({total_pct_h:.1f}%)')

✓ Figure P1 saved: fig_p1_dataset_overview.png
✓ Figure P2 saved: fig_p2_heatmap.png
  Congruent   : 13,151  (81.4%)
  Incongruent : 3,005  (18.6%)


In [4]:
print("=== Describe ===")
print(df['User_Contributions'].describe())

print("\n=== Value Counts (first 20) ===")
print(df['User_Contributions'].value_counts().sort_index().head(20))

plt.figure(figsize=(8,5))
plt.hist(df['User_Contributions'], bins=50)
plt.xlabel('User Contributions')
plt.ylabel('Frequency')
plt.title('Distribution of Reviewer Contributions')
plt.show()

bins = [-1, 5, 20, 100, float('inf')]
labels = ['0-5', '6-20', '21-100', '101+']

df['Tier_Check'] = pd.cut(df['User_Contributions'], bins=bins, labels=labels)

print("\n=== Tier Distribution (%) ===")
print(df['Tier_Check'].value_counts(normalize=True) * 100)

=== Describe ===
count    16156.000000
mean       191.624845
std        500.100421
min          1.000000
25%         18.000000
50%         54.000000
75%        155.000000
max       9010.000000
Name: User_Contributions, dtype: float64

=== Value Counts (first 20) ===
User_Contributions
1     151
2     251
3     284
4     309
5     298
6     258
7     259
8     268
9     240
10    263
11    228
12    249
13    219
14    181
15    205
16    179
17    175
18    161
19    137
20    164
Name: count, dtype: int64

=== Tier Distribution (%) ===
Tier_Check
21-100    37.249319
101+      35.027234
6-20      19.720228
0-5        8.003219
Name: proportion, dtype: float64


## 3. Feature Engineering

In [5]:
# ── 3a. Incongruent binary flag ─────────────────────────────────────────
df['Incongruent'] = (~(
    ((df['Rating_Class'] == 'Positive') & (df['Sentiment'] == 'POSITIVE')) |
    ((df['Rating_Class'] == 'Neutral')  & (df['Sentiment'] == 'NEUTRAL'))  |
    ((df['Rating_Class'] == 'Negative') & (df['Sentiment'] == 'NEGATIVE'))
)).astype(int)

# ── 3b. Full pattern labels ─────────────────────────────────────────────
def assign_pattern(row):
    rc  = row['Rating_Class']
    sen = row['Sentiment']
    if   rc == 'Positive' and sen == 'POSITIVE': return 'Congruent-Positive'
    elif rc == 'Neutral'  and sen == 'NEUTRAL' : return 'Congruent-Neutral'
    elif rc == 'Negative' and sen == 'NEGATIVE': return 'Congruent-Negative'
    elif rc == 'Neutral'  and sen == 'POSITIVE': return 'Conservative Rater'
    elif rc == 'Positive' and sen == 'NEUTRAL' : return 'Obligatory 5-Star'
    elif rc == 'Neutral'  and sen == 'NEGATIVE': return 'Frustrated Neutral'
    elif rc == 'Positive' and sen == 'NEGATIVE': return 'Polite Inflator'
    elif rc == 'Negative' and sen == 'NEUTRAL' : return 'Harsh Deflator'
    elif rc == 'Negative' and sen == 'POSITIVE': return 'Punitive Rater'
    else: return 'Unknown'

df['Pattern'] = df.apply(assign_pattern, axis=1)

# ── 3c. Reviewer Tier (fixed so 0 contributions are not silently lost) ──
df['Reviewer_Tier'] = df['User_Contributions'].apply(tier_label)

# ── 3d. Log-transform skewed continuous predictors ───────────────────────
df['log_review_length'] = np.log1p(df['Review_Length'])
df['log_review_delay']  = np.log1p(df['Review_Delay_Days'])

print('✓ Feature engineering complete.')
print(f'  Incongruent reviews : {df["Incongruent"].sum():,} ({df["Incongruent"].mean()*100:.1f}%)')
print(f'  Congruent reviews   : {(df["Incongruent"]==0).sum():,} ({(df["Incongruent"]==0).mean()*100:.1f}%)')
print(f'\n  Reviewer Tier distribution:')
print(df['Reviewer_Tier'].value_counts(dropna=False).sort_index().to_string())

✓ Feature engineering complete.
  Incongruent reviews : 3,005 (18.6%)
  Congruent reviews   : 13,151 (81.4%)

  Reviewer Tier distribution:
Reviewer_Tier
Active (21-100)    6018
Casual (6-20)      3186
Expert (101+)      5659
Novice (0-5)       1293


## 4. RQ1 — Prevalence and Full Typology of Directional Patterns

In [6]:
total     = len(df)
inc_count = df['Incongruent'].sum()
con_count = total - inc_count
inc_rate  = inc_count / total * 100

pattern_order = [
    'Conservative Rater', 'Obligatory 5-Star', 'Frustrated Neutral',
    'Polite Inflator', 'Harsh Deflator', 'Punitive Rater',
]

inc_df     = df[df['Incongruent'] == 1]
pat_counts = inc_df['Pattern'].value_counts()

print(f'Total reviews    : {total:,}')
print(f'Congruent        : {con_count:,} ({con_count/total*100:.1f}%)')
print(f'Incongruent      : {inc_count:,} ({inc_rate:.1f}%)')
print(f'~1 in every {int(round(1/df["Incongruent"].mean()))} reviews shows a mismatch.')
print(f'\nSix incongruent directional patterns:\n')
print(f'  {"Pattern":<22}  {"n":>6}  {"% of Incongruent":>18}  {"% of All Reviews":>18}')
print(f'  {"-"*70}')
for p in pattern_order:
    n   = pat_counts.get(p, 0)
    p_i = n / inc_count * 100
    p_a = n / total * 100
    print(f'  {p:<22}  {n:>6,}  {p_i:>17.1f}%  {p_a:>17.1f}%')
print(f'  {"-"*70}')
print(f'  {"TOTAL":<22}  {inc_count:>6,}  {100.0:>17.1f}%  {inc_rate:>17.1f}%')

# ── Figure 1: Prevalence doughnut + 6-pattern bar ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax1 = axes[0]
ax1.set_facecolor('white')
wedges, _ = ax1.pie(
    [con_count, inc_count], colors=[P['main'], P['acc']],
    startangle=90, wedgeprops=dict(width=0.52, edgecolor='white', linewidth=3))
ax1.text(0,  0.10, f'{inc_rate:.1f}%', ha='center', va='center',
         fontsize=24, fontweight='800', color=P['acc'])
ax1.text(0, -0.18, 'incongruent', ha='center', va='center', fontsize=11, color='#666')
ax1.text(0, -0.38, f'n = {inc_count:,} of {total:,}', ha='center',
         va='center', fontsize=9, color='#888')
ax1.legend(wedges, [f'Congruent ({con_count/total*100:.1f}%)',
                    f'Incongruent ({inc_rate:.1f}%)'],
           loc='lower center', bbox_to_anchor=(0.5, -0.06), frameon=False)
ax1.set_title('Overall Prevalence of Incongruence', pad=14, fontweight='700')

ax2      = axes[1]
counts_r = [pat_counts.get(p, 0) for p in pattern_order]
pcts_r   = [c / inc_count * 100 for c in counts_r]
bar_cols = [P['acc'], P['neu'], '#E8845C', '#C9506A', P['main'], P['grn']]
bars = ax2.barh(pattern_order, counts_r,
                color=bar_cols, height=0.62, edgecolor='white', linewidth=0.8)
for bar, cnt, pct in zip(bars, counts_r, pcts_r):
    ax2.text(bar.get_width() + 12, bar.get_y() + bar.get_height()/2,
             f'{cnt:,} ({pct:.1f}%)', va='center', fontsize=8.5, color='#333')
ax2.set_xlabel('Number of incongruent reviews')
ax2.set_xlim(0, max(counts_r) * 1.48)
ax2.set_title(f'All Six Directional Incongruence Patterns\n(n = {inc_count:,} incongruent reviews)',
              pad=10, fontweight='700')
plt.tight_layout(pad=2.5)
plt.savefig('fig1_rq1_prevalence_typology.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 1 saved: fig1_rq1_prevalence_typology.png')

Total reviews    : 16,156
Congruent        : 13,151 (81.4%)
Incongruent      : 3,005 (18.6%)
~1 in every 5 reviews shows a mismatch.

Six incongruent directional patterns:

  Pattern                      n    % of Incongruent    % of All Reviews
  ----------------------------------------------------------------------
  Conservative Rater       1,155               38.4%                7.1%
  Obligatory 5-Star          851               28.3%                5.3%
  Frustrated Neutral         509               16.9%                3.2%
  Polite Inflator            219                7.3%                1.4%
  Harsh Deflator             160                5.3%                1.0%
  Punitive Rater             111                3.7%                0.7%
  ----------------------------------------------------------------------
  TOTAL                    3,005              100.0%               18.6%
✓ Figure 1 saved: fig1_rq1_prevalence_typology.png


## 5. RQ2 — Incongruence by Location Type

In [7]:
lt_stats = df.groupby('Location_Type').agg(
    N           = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum'),
    Rate        = ('Incongruent', 'mean')
).sort_values('Rate', ascending=False)
lt_stats['Rate_Pct'] = (lt_stats['Rate'] * 100).round(2)
lt_stats['Congruent']= lt_stats['N'] - lt_stats['Incongruent']
lt_stats['Odds']     = lt_stats['Incongruent'] / lt_stats['Congruent']
ref_odds             = lt_stats.loc['National Parks', 'Odds']
lt_stats['OR']       = (lt_stats['Odds'] / ref_odds).round(2)

ct_lt                    = pd.crosstab(df['Location_Type'], df['Incongruent'])
chi2_lt, p_lt, dof_lt, _ = chi2_contingency(ct_lt)
n_lt                     = ct_lt.values.sum()
cramers_v                = np.sqrt(chi2_lt / (n_lt * (min(ct_lt.shape) - 1)))
mean_rate                = df['Incongruent'].mean() * 100

print(f'{"Location Type":<26} {"N":>6} {"Incongruent":>12} {"Rate (%)":>10} {"OR vs NP":>10}')
print('-'*68)
for lt, row in lt_stats.iterrows():
    print(f'{lt:<26} {int(row["N"]):>6,} {int(row["Incongruent"]):>12,} '
          f'{row["Rate_Pct"]:>9.1f}% {row["OR"]:>10.2f}')
print(f'\nChi-square: chi2({dof_lt}) = {chi2_lt:.2f}, p = {p_lt:.2e}')
print(f"Cramer's V = {cramers_v:.4f}")

# Figure 2
fig, ax = plt.subplots(figsize=(11, 6))
lp       = lt_stats.sort_values('Rate_Pct')
rate_vals= lp['Rate_Pct'].values
col_list = [P['acc'] if r >= 23 else (P['neu'] if r >= 19 else P['grn'])
            for r in rate_vals]
bars = ax.barh(lp.index, rate_vals, color=col_list,
               height=0.65, edgecolor='white', linewidth=0.8)
for bar, rate, OR in zip(bars, rate_vals, lp['OR'].values):
    ax.text(bar.get_width() + 0.25, bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}% (OR = {OR:.2f}x)', va='center', fontsize=9, color='#333')
ax.axvline(mean_rate, color='#777', linestyle='--', linewidth=1.3, zorder=5)
ax.text(mean_rate + 0.2, -0.7, f'Overall mean\n({mean_rate:.1f}%)',
        fontsize=8, color='#777', va='top')
ax.set_xlabel('Incongruence Rate (%)')
ax.set_xlim(0, 37)
ax.set_title(
    f'Incongruence Rate by Location Type\n'
    f"chi2({dof_lt}) = {chi2_lt:.1f}, p < 0.001, Cramer's V = {cramers_v:.3f} | Reference: National Parks",
    pad=10, fontweight='700')
ax.legend(handles=[
    Patch(facecolor=P['acc'], label='High (>= 23%)'),
    Patch(facecolor=P['neu'], label='Medium (19–23%)'),
    Patch(facecolor=P['grn'], label='Low (< 19%)'),
    mlines.Line2D([], [], color='#777', linestyle='--',
                  linewidth=1.3, label=f'Overall mean ({mean_rate:.1f}%)')],
    frameon=False, loc='lower right', fontsize=9)
plt.tight_layout(pad=2)
plt.savefig('fig2_rq2_location_type.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 2 saved: fig2_rq2_location_type.png')

Location Type                   N  Incongruent   Rate (%)   OR vs NP
--------------------------------------------------------------------
Museums                     1,525          401      26.3%       2.43
Bodies of Water               839          199      23.7%       2.12
Zoological Gardens            213           46      21.6%       1.88
Religious Sites             3,017          596      19.8%       1.68
Beaches                     2,110          401      19.0%       1.60
Waterfalls                    933          168      18.0%       1.50
Gardens                     1,354          235      17.4%       1.43
Farms                       1,884          312      16.6%       1.35
Nature & Wildlife Areas     1,557          256      16.4%       1.34
Historic Sites              1,519          237      15.6%       1.26
National Parks              1,205          154      12.8%       1.00

Chi-square: chi2(10) = 125.85, p = 3.27e-22
Cramer's V = 0.0883
✓ Figure 2 saved: fig2_rq2_location_ty

## 6. RQ3a — Bivariate Predictor Tests

- **Chi-square** for categorical predictors
- **Mann–Whitney U** for continuous predictors

These tests are descriptive screening tools only.  
A non-significant result means **insufficient evidence detected here**, not proof of no effect.

To reduce over-interpretation across multiple bivariate tests, this notebook now also reports **Benjamini–Hochberg FDR-adjusted q-values** alongside the raw p-values.


In [8]:
inc = df[df['Incongruent'] == 1]
con = df[df['Incongruent'] == 0]

# Reviewer Tier
ct_tier = pd.crosstab(df['Reviewer_Tier'], df['Incongruent'])
chi2_ti, p_ti, _, _ = chi2_contingency(ct_tier)
tier_stats = df.groupby('Reviewer_Tier', observed=True).agg(
    N=('Incongruent', 'count'),
    Incongruent=('Incongruent', 'sum'),
    Rate=('Incongruent', 'mean')
)
tier_stats['Rate_Pct'] = (tier_stats['Rate'] * 100).round(1)
tier_stats['Congruent'] = tier_stats['N'] - tier_stats['Incongruent']
tier_stats['Odds'] = tier_stats['Incongruent'] / tier_stats['Congruent']
novice_odds = tier_stats.loc['Novice (0-5)', 'Odds']
tier_stats['OR'] = (tier_stats['Odds'] / novice_odds).round(2)

# Review Length
u_len, p_len = mannwhitneyu(inc['log_review_length'], con['log_review_length'], alternative='two-sided')

# Review Delay
u_del, p_del = mannwhitneyu(inc['log_review_delay'], con['log_review_delay'], alternative='two-sided')

# Province
ct_prov = pd.crosstab(df['Province'], df['Incongruent'])
chi2_pv, p_pv, dof_pv, _ = chi2_contingency(ct_prov)
prov_rates = df.groupby('Province')['Incongruent'].mean().mul(100).round(1).sort_values(ascending=False)

# Travel Year
u_yr, p_yr = mannwhitneyu(inc['Travel_Date_Year'], con['Travel_Date_Year'], alternative='two-sided')

# Multiple-testing correction across the main bivariate screening tests
bivar_tests = pd.DataFrame({
    'Test': ['Reviewer Tier', 'Review Length', 'Review Delay', 'Province', 'Travel Year'],
    'raw_p': [p_ti, p_len, p_del, p_pv, p_yr]
})
bivar_tests['q_value'] = multipletests(bivar_tests['raw_p'], method='fdr_bh')[1]
bivar_tests['FDR_sig'] = bivar_tests['q_value'] < 0.05

print('Reviewer Tier:')
print(tier_stats[['N', 'Incongruent', 'Rate_Pct', 'OR']].to_string())
print(f'chi2(3) = {chi2_ti:.2f}, raw p = {p_ti:.2e}')
print(f'Expert reviewers are {tier_stats.loc["Expert (101+)","OR"]:.2f}x more likely to be incongruent than Novices.\n')

print(f'Review Length: Median incongruent = {inc["Review_Length"].median():.0f} chars | Congruent = {con["Review_Length"].median():.0f} chars')
print(f'Mann–Whitney raw p = {p_len:.4f}\n')

print(f'Review Delay: Median incongruent = {inc["Review_Delay_Days"].median():.0f} days | Congruent = {con["Review_Delay_Days"].median():.0f} days')
print(f'Mann–Whitney raw p = {p_del:.4f}')
print('Interpretation: no statistically significant evidence was detected here in the bivariate screen.\n')

print(f'Province: chi2({dof_pv}) = {chi2_pv:.2f}, raw p = {p_pv:.4f}')
print(prov_rates.to_string())

print(f'\nTravel Year: Mann–Whitney raw p = {p_yr:.4f}')

print('\nBivariate screening summary with FDR adjustment:')
print(bivar_tests.to_string(index=False))

# Figure 3: Reviewer Tier bars + Review Length violin
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax1 = axes[0]
ts = tier_stats.sort_index()
tcols = [P['grn'], P['light'], P['main'], P['acc']]
b = ax1.bar(ts.index.astype(str), ts['Rate_Pct'],
            color=tcols, edgecolor='white', width=0.6, linewidth=0.8)
for bar, rate, OR in zip(b, ts['Rate_Pct'], ts['OR']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.35,
             f'{rate}%\nOR = {OR:.2f}x', ha='center', va='bottom', fontsize=8.5, fontweight='500')
ax1.axhline(mean_rate, color='#777', linestyle='--', linewidth=1.2,
            label=f'Overall mean ({mean_rate:.1f}%)')
ax1.set_ylim(0, 28)
ax1.set_xlabel('Reviewer Tier')
ax1.set_ylabel('Incongruence Rate (%)')
ax1.set_title(f'Incongruence by Reviewer Expertise\nchi2(3) = {chi2_ti:.1f}, raw p < 0.001 | Ref: Novice',
              pad=8, fontweight='700')
ax1.legend(frameon=False)

ax2 = axes[1]
cap_val = df['Review_Length'].quantile(0.97)
plot_df = df[df['Review_Length'] <= cap_val].copy()
parts = ax2.violinplot(
    [plot_df[plot_df['Incongruent'] == 0]['Review_Length'],
     plot_df[plot_df['Incongruent'] == 1]['Review_Length']],
    positions=[0, 1], showmedians=True, showextrema=False)
for pc, col in zip(parts['bodies'], [P['main'], P['acc']]):
    pc.set_facecolor(col); pc.set_alpha(0.68)
parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Congruent', 'Incongruent'])
ax2.set_ylabel('Review Length (characters)')
diff = int(inc['Review_Length'].median() - con['Review_Length'].median())
ax2.set_title(
    f'Review Length by Congruence\nMann–Whitney raw p = {p_len:.4f} | Median diff = +{diff} chars',
    pad=8, fontweight='700')
ax2.text(0.5, 0.93, f'Incongruent reviews are {diff} chars longer (median)',
         ha='center', transform=ax2.transAxes, fontsize=9, color='#444',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f0', edgecolor='#ddd'))
plt.tight_layout(pad=2.5)
plt.savefig('fig3_rq3_reviewer_tier_length.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 3 saved: fig3_rq3_reviewer_tier_length.png')


Reviewer Tier:
                    N  Incongruent  Rate_Pct    OR
Reviewer_Tier                                     
Active (21-100)  6018         1142      19.0  1.69
Casual (6-20)    3186          494      15.5  1.33
Expert (101+)    5659         1212      21.4  1.97
Novice (0-5)     1293          157      12.1  1.00
chi2(3) = 85.99, raw p = 1.59e-18
Expert reviewers are 1.97x more likely to be incongruent than Novices.

Review Length: Median incongruent = 296 chars | Congruent = 279 chars
Mann–Whitney raw p = 0.0011

Review Delay: Median incongruent = 25 days | Congruent = 25 days
Mann–Whitney raw p = 0.7503
Interpretation: no statistically significant evidence was detected here in the bivariate screen.

Province: chi2(7) = 49.10, raw p = 0.0000
Province
Western Province          20.6
Northern Province         20.0
North Central Province    20.0
Central Province          19.9
Sabaragamuwa Province     16.8
Uva Province              16.7
Eastern Province          16.3
Southern Provin

## 7. RQ3b — Multicollinearity Check (VIF)

For VIF we use the design matrix intended for the explanatory model, with reviewer tier represented through dummies.

In [9]:
df_lr = df.dropna(subset=['Reviewer_Tier']).copy()

# Explanatory dummies
lt_dummies = pd.get_dummies(df_lr['Location_Type'], prefix='LT', drop_first=False)
if 'LT_National Parks' in lt_dummies.columns:
    lt_dummies = lt_dummies.drop(columns=['LT_National Parks'])

pv_dummies = pd.get_dummies(df_lr['Province'], prefix='PV', drop_first=False)
if 'PV_Southern Province' in pv_dummies.columns:
    pv_dummies = pv_dummies.drop(columns=['PV_Southern Province'])

tier_dummies = pd.get_dummies(df_lr['Reviewer_Tier'], prefix='Tier', drop_first=False)
if 'Tier_Novice (0-5)' in tier_dummies.columns:
    tier_dummies = tier_dummies.drop(columns=['Tier_Novice (0-5)'])

X_vif = pd.concat([
    lt_dummies.reset_index(drop=True),
    pv_dummies.reset_index(drop=True),
    tier_dummies.reset_index(drop=True),
    df_lr.assign(Travel_Year_c=df_lr['Travel_Date_Year'] - df_lr['Travel_Date_Year'].mean(),
             Travel_Year_c2=(df_lr['Travel_Date_Year'] - df_lr['Travel_Date_Year'].mean())**2)
     [['log_review_length', 'log_review_delay', 'Travel_Year_c', 'Travel_Year_c2']].reset_index(drop=True),
], axis=1).astype(float)

y = df_lr['Incongruent'].reset_index(drop=True)

print(f'Feature matrix for inference: {X_vif.shape[0]:,} observations x {X_vif.shape[1]} predictors')
print('Travel year is modeled with a centered linear and quadratic term.')
print(f'Outcome rate: {y.mean()*100:.1f}% incongruent')

def compute_vif(X_df):
    Xv       = X_df.values
    vif_vals = []
    for i in range(Xv.shape[1]):
        y_col  = Xv[:, i]
        X_rest = np.delete(Xv, i, axis=1)
        r2     = LinearRegression().fit(X_rest, y_col).score(X_rest, y_col)
        vif_j  = 1.0 / (1.0 - r2) if r2 < 0.9999 else np.inf
        vif_vals.append(vif_j)
    return vif_vals

vif_values = compute_vif(X_vif)
vif_df = pd.DataFrame({'Predictor': X_vif.columns, 'VIF': vif_values}).sort_values('VIF', ascending=False).reset_index(drop=True)
vif_df['Status'] = vif_df['VIF'].apply(lambda v: 'HIGH' if v > 10 else ('MODERATE' if v > 5 else 'OK'))

print(f'{"Predictor":<40} {"VIF":>8}  Status')
print('-'*60)
for _, row in vif_df.iterrows():
    flag = '  <-- CHECK' if row['VIF'] > 5 else ''
    print(f'{row["Predictor"]:<40} {row["VIF"]:>8.3f}  {row["Status"]}{flag}')
print('-'*60)
print(f'{"Max VIF":<40} {vif_df["VIF"].max():>8.3f}')
print(f'{"Mean VIF":<40} {vif_df["VIF"].mean():>8.3f}')

Feature matrix for inference: 16,156 observations x 24 predictors
Travel year is modeled with a centered linear and quadratic term.
Outcome rate: 18.6% incongruent
Predictor                                     VIF  Status
------------------------------------------------------------
Tier_Expert (101+)                          3.663  OK
Tier_Active (21-100)                        3.653  OK
LT_Religious Sites                          3.267  OK
LT_Farms                                    3.007  OK
LT_Beaches                                  2.895  OK
Tier_Casual (6-20)                          2.813  OK
PV_Central Province                         2.682  OK
PV_North Central Province                   2.653  OK
LT_Museums                                  2.581  OK
LT_Gardens                                  2.510  OK
LT_Historic Sites                           2.349  OK
LT_Nature & Wildlife Areas                  2.282  OK
LT_Waterfalls                               2.268  OK
LT_Bodies of Wa

## 8. RQ3c — Model 1A: Logistic Regression Evaluation Model (Leakage-Free)

This model exists **only** to estimate predictive discrimination fairly.

### Design
- one **80/20 stratified split**
- **Pipeline(StandardScaler, LogisticRegression)** so CV scaling happens fold-by-fold
- `class_weight='balanced'` retained for prediction-oriented comparison
- test AUC computed using a model trained on **training data only**
- includes centered linear and quadratic year terms to relax a strict linear time assumption


In [10]:
# Shared 80/20 split used by both evaluation models
X_eval = X_vif.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_eval, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train set : {len(y_train):,} reviews | {y_train.mean()*100:.1f}% incongruent')
print(f'Test set  : {len(y_test):,} reviews | {y_test.mean()*100:.1f}% incongruent')
print('✓ Stratified split created. This same test set will be used for LR and SVM evaluation.')

Train set : 12,924 reviews | 18.6% incongruent
Test set  : 3,232 reviews | 18.6% incongruent
✓ Stratified split created. This same test set will be used for LR and SVM evaluation.


In [11]:
C_grid = [0.01, 0.1, 1.0, 10.0]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
print('Model 1A — 5-Fold CV Grid Search on TRAIN set (Pipeline, no leakage)')
print(f'{"C":>8}  {"Mean CV AUC":>12}  {"Std":>8}  {"Min":>8}  {"Max":>8}')
print('-' * 52)

for C_val in C_grid:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            C=C_val, class_weight='balanced',
            max_iter=5000, random_state=42, solver='lbfgs'
        ))
    ])
    cv_aucs = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    cv_results.append({'C': C_val, 'mean_auc': cv_aucs.mean(),
                       'std_auc': cv_aucs.std(), 'fold_aucs': cv_aucs})
    print(f'{C_val:>8.2f}  {cv_aucs.mean():>12.4f}  {cv_aucs.std():>8.4f}  '
          f'{cv_aucs.min():>8.4f}  {cv_aucs.max():>8.4f}')

cv_df = pd.DataFrame(cv_results)
best_idx = cv_df['mean_auc'].idxmax()
best_C = cv_df.loc[best_idx, 'C']
best_mean_cv = cv_df.loc[best_idx, 'mean_auc']
best_std_cv = cv_df.loc[best_idx, 'std_auc']

# Final evaluation model trained on TRAIN only
lr_eval = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        C=best_C, class_weight='balanced',
        max_iter=5000, random_state=42, solver='lbfgs'
    ))
])
lr_eval.fit(X_train, y_train)

y_proba_test_m1 = lr_eval.predict_proba(X_test)[:, 1]
auc_test_m1 = roc_auc_score(y_test, y_proba_test_m1)

print('\n' + '=' * 74)
print('MODEL 1A — LR Evaluation Model (balanced, leakage-free)')
print('=' * 74)
print(f'Best C               : {best_C}')
print(f'Mean CV AUC (5-fold) : {best_mean_cv:.4f} ± {best_std_cv:.4f}')
print(f'Held-out test AUC    : {auc_test_m1:.4f}')

Model 1A — 5-Fold CV Grid Search on TRAIN set (Pipeline, no leakage)
       C   Mean CV AUC       Std       Min       Max
----------------------------------------------------
    0.01        0.5883    0.0101    0.5704    0.6017
    0.10        0.5889    0.0094    0.5731    0.6026
    1.00        0.5890    0.0093    0.5734    0.6026
   10.00        0.5890    0.0093    0.5734    0.6026

MODEL 1A — LR Evaluation Model (balanced, leakage-free)
Best C               : 1.0
Mean CV AUC (5-fold) : 0.5890 ± 0.0093
Held-out test AUC    : 0.5840


## 9. RQ3d — Model 1B: Explanatory Logistic Regression with `statsmodels`

This is the **inference model** used for:
- coefficients
- odds ratios
- 95% confidence intervals
- p-values

### Important note
For inference, the model is fit **without `class_weight='balanced'`**.  
That weighting is helpful for prediction, but it distorts classical coefficient interpretation.

In [12]:
# Build statsmodels design matrix (categorical dummies + continuous predictors)
X_sm = X_vif.copy()
X_sm_const = sm.add_constant(X_sm, has_constant='add')

logit_model = sm.Logit(y, X_sm_const)
logit_result = logit_model.fit(disp=False)

coef = logit_result.params
se = logit_result.bse
pvals = logit_result.pvalues
ci = logit_result.conf_int()
ci.columns = ['CI_lo_logodds', 'CI_hi_logodds']

results_m1 = pd.DataFrame({
    'Predictor': coef.index,
    'Coef': coef.values,
    'SE': se.values,
    'OR': np.exp(coef.values),
    'CI_lo': np.exp(ci['CI_lo_logodds'].values),
    'CI_hi': np.exp(ci['CI_hi_logodds'].values),
    'p_value': pvals.values
})

# Drop intercept from table/plot
results_m1 = results_m1[results_m1['Predictor'] != 'const'].copy()
results_m1['Sig'] = results_m1['p_value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
)
results_m1 = results_m1.sort_values('OR', ascending=False).reset_index(drop=True)

sig_preds = results_m1.loc[results_m1['p_value'] < 0.05, 'Predictor'].tolist()
ns_preds  = results_m1.loc[results_m1['p_value'] >= 0.05, 'Predictor'].tolist()

print('=' * 80)
print('MODEL 1B — statsmodels Logit (Inference model)')
print('=' * 80)
print(f'Pseudo R-squared      : {logit_result.prsquared:.4f}')
print(f'LLR p-value           : {logit_result.llr_pvalue:.4e}')
print(f'Significant predictors: {len(sig_preds)}')
print()
print(f'{"Predictor":<40} {"OR":>8} {"95% CI":>22} {"p":>10} {"Sig":>5}')
print('-' * 90)
for _, row in results_m1.iterrows():
    ci_txt = f'[{row["CI_lo"]:.3f}, {row["CI_hi"]:.3f}]'
    print(f'{row["Predictor"]:<40} {row["OR"]:>8.3f} {ci_txt:>22} {row["p_value"]:>10.4f} {row["Sig"]:>5}')

MODEL 1B — statsmodels Logit (Inference model)
Pseudo R-squared      : 0.0203
LLR p-value           : 2.1382e-52
Significant predictors: 19

Predictor                                      OR                 95% CI          p   Sig
------------------------------------------------------------------------------------------
LT_Museums                                  2.386         [1.895, 3.005]     0.0000   ***
LT_Beaches                                  2.171         [1.734, 2.717]     0.0000   ***
LT_Bodies of Water                          2.022         [1.577, 2.592]     0.0000   ***
PV_Northern Province                        1.787         [1.348, 2.368]     0.0001   ***
Tier_Expert (101+)                          1.740         [1.450, 2.089]     0.0000   ***
LT_Zoological Gardens                       1.725         [1.145, 2.598]     0.0091    **
PV_Sabaragamuwa Province                    1.641         [1.236, 2.179]     0.0006   ***
PV_North Central Province                   1.60

In [13]:
# Figure 4 — Forest plot from valid statsmodels inference
label_map = {
    'LT_Beaches'                         : 'Beaches (vs. Nat. Parks)',
    'LT_Bodies of Water'                 : 'Bodies of Water (vs. Nat. Parks)',
    'LT_Farms'                           : 'Farms (vs. Nat. Parks)',
    'LT_Gardens'                         : 'Gardens (vs. Nat. Parks)',
    'LT_Historic Sites'                  : 'Historic Sites (vs. Nat. Parks)',
    'LT_Museums'                         : 'Museums (vs. Nat. Parks)',
    'LT_Nature & Wildlife Areas'         : 'Nature & Wildlife (vs. Nat. Parks)',
    'LT_Religious Sites'                 : 'Religious Sites (vs. Nat. Parks)',
    'LT_Waterfalls'                      : 'Waterfalls (vs. Nat. Parks)',
    'LT_Zoological Gardens'              : 'Zoological Gardens (vs. Nat. Parks)',
    'PV_Central Province'                : 'Central Province (vs. Southern)',
    'PV_Eastern Province'                : 'Eastern Province (vs. Southern)',
    'PV_North Central Province'          : 'North Central (vs. Southern)',
    'PV_Northern Province'               : 'Northern Province (vs. Southern)',
    'PV_Sabaragamuwa Province'           : 'Sabaragamuwa (vs. Southern)',
    'PV_Uva Province'                    : 'Uva Province (vs. Southern)',
    'PV_Western Province'                : 'Western Province (vs. Southern)',
    'Tier_Casual (6-20)'                 : 'Casual tier (vs. Novice)',
    'Tier_Active (21-100)'               : 'Active tier (vs. Novice)',
    'Tier_Expert (101+)'                 : 'Expert tier (vs. Novice)',
    'log_review_length'                  : 'Review Length (log chars)',
    'log_review_delay'                   : 'Review Delay (log days)',
    'Travel_Date_Year'                   : 'Travel Year',
}
rs = results_m1.copy()
rs['label'] = rs['Predictor'].map(label_map).fillna(rs['Predictor'])

fig, ax = plt.subplots(figsize=(11, max(8, len(rs)*0.42)))
ypos = np.arange(len(rs))

for i, (_, row) in enumerate(rs.iterrows()):
    sig = row['p_value'] < 0.05
    col = P['acc'] if sig else '#AAAAAA'
    ax.plot([row['CI_lo'], row['CI_hi']], [i, i], color=col, linewidth=1.6, zorder=2, solid_capstyle='round')
    ax.plot([row['CI_lo']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    ax.plot([row['CI_hi']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    ax.scatter(row['OR'], i, color=col, s=65, zorder=3, edgecolors='white', linewidth=0.8)
    ax.text(max(row['CI_hi'], row['OR']) + 0.015, i,
            f'{row["OR"]:.3f} {row["Sig"]}', va='center', fontsize=7.5,
            color=P['acc'] if sig else '#999')

ax.axvline(1.0, color='#555', linestyle='--', linewidth=1.3, zorder=1)
ax.set_yticks(ypos)
ax.set_yticklabels(rs['label'], fontsize=8)
ax.set_xlabel('Odds Ratio (95% Confidence Interval)', fontsize=10)
ax.set_title(
    f'Model 1B — Explanatory Logistic Regression\n'
    f'Inference from statsmodels | n = {len(y):,} | Ref: National Parks, Novice, Southern Province',
    pad=10, fontweight='700')
ax.legend(handles=[
    mlines.Line2D([], [], marker='o', color=P['acc'], markersize=8, linestyle='none', label='Significant (p < 0.05)'),
    mlines.Line2D([], [], marker='o', color='#AAA', markersize=8, linestyle='none', label='Not significant'),
    mlines.Line2D([], [], color='#555', linestyle='--', linewidth=1.5, label='OR = 1.0')],
    frameon=False, loc='lower right', fontsize=9)
plt.tight_layout(pad=2)
plt.savefig('fig4_rq3_forest_plot.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 4 saved: fig4_rq3_forest_plot.png')

✓ Figure 4 saved: fig4_rq3_forest_plot.png


In [14]:
# Figure 5 — ROC curve from evaluation model only
fig, ax = plt.subplots(figsize=(6, 5.5))
fpr_test_m1, tpr_test_m1, _ = roc_curve(y_test, y_proba_test_m1)
ax.plot(fpr_test_m1, tpr_test_m1, color=P['acc'], linewidth=2.2,
        label=f'Model 1A LR (Test AUC = {auc_test_m1:.4f})')
ax.plot([0,1],[0,1], color='#bbb', linestyle=':', linewidth=1.3,
        label='Random classifier (AUC = 0.500)')
ax.fill_between(fpr_test_m1, tpr_test_m1, alpha=0.07, color=P['acc'])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Model 1A (Leakage-Free LR Evaluation Model)', pad=8, fontweight='700')
ax.legend(frameon=False, fontsize=9)
ax.text(0.55, 0.14,
        f'CV AUC = {best_mean_cv:.4f} ± {best_std_cv:.4f}\nTest AUC = {auc_test_m1:.4f}\nn = {len(y_test):,}',
        transform=ax.transAxes, fontsize=9, color='#444',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f0', edgecolor='#ccc'))
plt.tight_layout()
plt.savefig('fig5_roc_m1.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 5 saved: fig5_roc_m1.png')

✓ Figure 5 saved: fig5_roc_m1.png


## 10. RQ3e — Model 2: Tuned SVM with RBF Kernel

This section is now phrased correctly:

- A better SVM AUC can **suggest** nonlinear structure
- It does **not by itself prove** nonlinearity
- The comparison is made fairer by **tuning SVM on the training set**

In [15]:
svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        class_weight='balanced',
        probability=True,
        random_state=42
    ))
])

param_grid_svm = {
    'svm__C': [0.1, 1.0, 10.0],
    'svm__gamma': ['scale', 0.1, 0.01]
}

svm_search = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid_svm,
    scoring='roc_auc',
    cv=skf,
    n_jobs=-1,
    refit=True
)

print('Fitting tuned SVM on TRAIN set...')
svm_search.fit(X_train, y_train)

best_svm = svm_search.best_estimator_
best_svm_params = svm_search.best_params_
best_svm_cv_auc = svm_search.best_score_

y_proba_test_svm = best_svm.predict_proba(X_test)[:, 1]
auc_test_svm = roc_auc_score(y_test, y_proba_test_svm)

auc_delta = auc_test_svm - auc_test_m1
pct_gain = auc_delta / auc_test_m1 * 100

print('\n' + '=' * 68)
print('MODEL 2 — Tuned SVM RBF')
print('=' * 68)
print(f'Best params          : {best_svm_params}')
print(f'Mean CV AUC (5-fold) : {best_svm_cv_auc:.4f}')
print(f'Test AUC             : {auc_test_svm:.4f}')
print(f'AUC delta vs LR      : {auc_delta:+.4f} ({pct_gain:+.1f}% relative)')
if auc_delta > 0.01:
    print('Interpretation: The higher SVM AUC suggests that some nonlinear structure may be present.')
elif auc_delta > 0:
    print('Interpretation: The marginal improvement suggests at most mild nonlinear structure.')
else:
    print('Interpretation: The tuned SVM does not improve on LR, so there is little evidence of meaningful nonlinear gain here.')

Fitting tuned SVM on TRAIN set...

MODEL 2 — Tuned SVM RBF
Best params          : {'svm__C': 1.0, 'svm__gamma': 'scale'}
Mean CV AUC (5-fold) : 0.6098
Test AUC             : 0.6205
AUC delta vs LR      : +0.0365 (+6.2% relative)
Interpretation: The higher SVM AUC suggests that some nonlinear structure may be present.


In [16]:
# Figure 6 — ROC comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Panel A
ax1 = axes[0]
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_proba_test_svm)
ax1.plot(fpr_test_m1, tpr_test_m1, color=P['acc'], linewidth=2.2,
         label=f'Model 1A — LR (AUC = {auc_test_m1:.4f})')
ax1.plot(fpr_svm, tpr_svm, color=P['purple'], linewidth=2.0, linestyle='--',
         label=f'Model 2 — Tuned SVM (AUC = {auc_test_svm:.4f})')
ax1.plot([0,1],[0,1], color='#bbb', linestyle=':', linewidth=1.3, label='Random classifier')
ax1.fill_between(fpr_test_m1, tpr_test_m1, alpha=0.05, color=P['acc'])
ax1.fill_between(fpr_svm, tpr_svm, alpha=0.05, color=P['purple'])
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves — LR vs Tuned SVM\nSame held-out 20% test set', pad=8, fontweight='700')
ax1.legend(frameon=False, fontsize=9)
ax1.text(0.52, 0.08,
         f'AUC delta = {auc_delta:+.4f}\n({pct_gain:+.1f}% relative gain from SVM)',
         transform=ax1.transAxes, fontsize=9, color='#444',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f0', edgecolor='#ccc'))

# Panel B
ax2 = axes[1]
labels = ['Model 1A\nLR', 'Model 2\nTuned SVM']
auc_vals = [auc_test_m1, auc_test_svm]
colors = [P['acc'], P['purple']]
bars = ax2.bar(labels, auc_vals, color=colors, edgecolor='white', width=0.45, zorder=3)
for bar, val in zip(bars, auc_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', fontsize=11, fontweight='700')
ax2.axhline(0.5, color='#999', linestyle=':', linewidth=1.3,
            label='Random baseline (0.500)', zorder=2)
ax2.set_ylim(0.45, max(auc_vals) + 0.06)
ax2.set_ylabel('AUC-ROC (Test Set)')
ax2.set_title(
    f'AUC Comparison — Identical Test Set\n'
    f'SVM gain over LR: {auc_delta:+.4f} ({pct_gain:+.1f}%)',
    pad=8, fontweight='700')
ax2.legend(frameon=False, fontsize=9)
ax2.text(0.5, 0.15,
         'SVM used as a secondary benchmark;\nLR retained for interpretability',
         ha='center', transform=ax2.transAxes, fontsize=9,
         color='#555', style='italic')

plt.tight_layout(pad=2.5)
plt.savefig('fig6_model_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print('✓ Figure 6 saved: fig6_model_comparison.png')

✓ Figure 6 saved: fig6_model_comparison.png


## 11. Supplementary — Pattern Composition by Reviewer Tier
This section is unchanged in logic, but its interpretation is kept descriptive rather than over-causal.

In [17]:
tier_pat = (df[df['Incongruent']==1]
            .groupby(['Reviewer_Tier','Pattern'], observed=True)
            .size().unstack(fill_value=0))
tier_pat_pct = (tier_pat.div(tier_pat.sum(axis=1), axis=0) * 100).round(1)
disp_cols = [c for c in pattern_order if c in tier_pat_pct.columns]
print('Pattern distribution within incongruent reviews by reviewer tier (%):')
print(tier_pat_pct[disp_cols].to_string())
print()
print('Descriptive note: If Conservative Rater increases monotonically from Novice to Expert,')
print('that pattern is consistent with a calibration-style explanation, though not by itself causal proof.')

Pattern distribution within incongruent reviews by reviewer tier (%):
Pattern          Conservative Rater  Obligatory 5-Star  Frustrated Neutral  Polite Inflator  Harsh Deflator  Punitive Rater
Reviewer_Tier                                                                                                              
Active (21-100)                38.4               27.6                17.5              7.9             5.0             3.7
Casual (6-20)                  37.2               25.1                16.6              8.5             7.1             5.5
Expert (101+)                  40.4               30.3                16.2              6.2             4.2             2.7
Novice (0-5)                   27.4               28.7                19.7              7.6            10.8             5.7

Descriptive note: If Conservative Rater increases monotonically from Novice to Expert,
that pattern is consistent with a calibration-style explanation, though not by itself causal proof

## 12. Full Results Summary
This section now clearly separates:
- **predictive evaluation** from
- **statistical inference**

It also reports the bivariate screening with **FDR-adjusted q-values** and uses a more flexible time specification in the inference model via centered linear and quadratic year terms.


In [18]:
print('=' * 72)
print('FULL RESULTS — SENTIMENT–RATING INCONGRUENCE IN SRI LANKA')
print('=' * 72)

print(f'\nRQ1: Prevalence and Typology')
print(f'  Overall incongruence rate : {inc_rate:.1f}% ({inc_count:,} / {total:,})')
print(f'  ~1 in {int(round(1/df["Incongruent"].mean()))} reviews is incongruent.')
print(f'\n  Pattern                  n        % of Inc.   % of All')
print('  ' + '-'*58)
for p in pattern_order:
    n  = pat_counts.get(p, 0)
    pi = n / inc_count * 100
    pa = n / total * 100
    print(f'  {p:<22}  {n:>5,}  {pi:>8.1f}%   {pa:>7.1f}%')

print(f'\nRQ2: Location Type')
print(f'  chi2({dof_lt}) = {chi2_lt:.2f}, p < 0.001, Cramér\'s V = {cramers_v:.4f}')
print(f'  Highest : Museums        {lt_stats.loc["Museums","Rate_Pct"]:.1f}% (OR = {lt_stats.loc["Museums","OR"]:.2f}x)')
print(f'  Lowest  : National Parks {lt_stats.loc["National Parks","Rate_Pct"]:.1f}% (reference)')

print(f'\nVIF Check')
print(f'  Max VIF = {vif_df["VIF"].max():.3f} | Mean VIF = {vif_df["VIF"].mean():.3f}')

print(f'\nRQ3: Bivariate Tests')
print(f'  Reviewer Tier chi2(3) = {chi2_ti:.2f}, raw p < 0.001')
print(f'    Expert vs Novice OR = {tier_stats.loc["Expert (101+)","OR"]:.2f}x')
print(f'  Review Length Mann–Whitney raw p = {p_len:.4f}')
print(f'    Incongruent median = {int(inc["Review_Length"].median())} chars | Congruent = {int(con["Review_Length"].median())} chars')
print(f'  Review Delay Mann–Whitney raw p = {p_del:.4f}')
print(f'  Province chi2({dof_pv}) = {chi2_pv:.2f}, raw p = {p_pv:.4f}')
print(f'  Travel Year Mann–Whitney raw p = {p_yr:.4f}')
print('  FDR-adjusted bivariate q-values:')
print(bivar_tests.to_string(index=False))

print(f'\nRQ3: Model 1A — LR Evaluation Model')
print(f'  Mean CV AUC (5-fold): {best_mean_cv:.4f} ± {best_std_cv:.4f}')
print(f'  Held-out test AUC   : {auc_test_m1:.4f}')

print(f'\nRQ3: Model 1B — Explanatory Logit')
print(f'  Pseudo R-squared    : {logit_result.prsquared:.4f}')
print(f'  Significant predictors: {len(sig_preds)}')
print(f'  Non-significant       : {", ".join(ns_preds)}')

print(f'\nRQ3: Model 2 — Tuned SVM RBF')
print(f'  Mean CV AUC (5-fold): {best_svm_cv_auc:.4f}')
print(f'  Held-out test AUC   : {auc_test_svm:.4f}')
print(f'  AUC delta vs LR     : {auc_delta:+.4f}')
if auc_delta > 0.01:
    print('  Verdict: The SVM improvement suggests some nonlinear structure may remain beyond the linear LR boundary.')
elif auc_delta > 0:
    print('  Verdict: The small SVM gain suggests at most mild nonlinearity.')
else:
    print('  Verdict: The tuned SVM does not improve on LR, so nonlinear gain appears limited.')
print('=' * 72)

FULL RESULTS — SENTIMENT–RATING INCONGRUENCE IN SRI LANKA

RQ1: Prevalence and Typology
  Overall incongruence rate : 18.6% (3,005 / 16,156)
  ~1 in 5 reviews is incongruent.

  Pattern                  n        % of Inc.   % of All
  ----------------------------------------------------------
  Conservative Rater      1,155      38.4%       7.1%
  Obligatory 5-Star         851      28.3%       5.3%
  Frustrated Neutral        509      16.9%       3.2%
  Polite Inflator           219       7.3%       1.4%
  Harsh Deflator            160       5.3%       1.0%
  Punitive Rater            111       3.7%       0.7%

RQ2: Location Type
  chi2(10) = 125.85, p < 0.001, Cramér's V = 0.0883
  Highest : Museums        26.3% (OR = 2.43x)
  Lowest  : National Parks 12.8% (reference)

VIF Check
  Max VIF = 3.663 | Mean VIF = 2.186

RQ3: Bivariate Tests
  Reviewer Tier chi2(3) = 85.99, raw p < 0.001
    Expert vs Novice OR = 1.97x
  Review Length Mann–Whitney raw p = 0.0011
    Incongruent median = 2

In [19]:
import os
figures = [
    'fig_p1_dataset_overview.png',
    'fig_p2_heatmap.png',
    'fig1_rq1_prevalence_typology.png',
    'fig2_rq2_location_type.png',
    'fig3_rq3_reviewer_tier_length.png',
    'fig4_rq3_forest_plot.png',
    'fig5_roc_m1.png',
    'fig6_model_comparison.png',
]
print('Output figures:')
for f in figures:
    status = '✓ SAVED' if os.path.exists(f) else '✗ MISSING'
    print(f'  [{status}] {f}')
print('\n✓ Corrected analysis notebook complete.')
print(f'  Model 1A LR — CV AUC   : {best_mean_cv:.4f} ± {best_std_cv:.4f}')
print(f'  Model 1A LR — Test AUC : {auc_test_m1:.4f}')
print(f'  Model 2 SVM — Test AUC : {auc_test_svm:.4f}')
print(f'  AUC delta (SVM - LR)   : {auc_delta:+.4f}')

Output figures:
  [✓ SAVED] fig_p1_dataset_overview.png
  [✓ SAVED] fig_p2_heatmap.png
  [✓ SAVED] fig1_rq1_prevalence_typology.png
  [✓ SAVED] fig2_rq2_location_type.png
  [✓ SAVED] fig3_rq3_reviewer_tier_length.png
  [✓ SAVED] fig4_rq3_forest_plot.png
  [✓ SAVED] fig5_roc_m1.png
  [✓ SAVED] fig6_model_comparison.png

✓ Corrected analysis notebook complete.
  Model 1A LR — CV AUC   : 0.5890 ± 0.0093
  Model 1A LR — Test AUC : 0.5840
  Model 2 SVM — Test AUC : 0.6205
  AUC delta (SVM - LR)   : +0.0365
